In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os
import requests
import re
# %%


    
    

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'BZ CBBEL'
 
print(f"Running {regulatorName} Web Scraping Tool v.1.2")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


#os.chdir(scriptfolder)
#print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))


Running BZ CBBEL Web Scraping Tool v.1.2


In [3]:

#------------------------------------------------ Begin_chromedriver ----------------------------------------

chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
		"plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()




In [7]:

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={
     'BZ CBBEL 1': 'https://www.centralbank.org.bz/home/core-functions/prudential-supervision/domestic-banks',
     'BZ CBBEL 2': 'https://www.centralbank.org.bz/home/core-functions/prudential-supervision/international-banks',
     'BZ CBBEL 3': 'https://www.centralbank.org.bz/home/core-functions/prudential-supervision/other-financial-institutions',
     'BZ CBBEL 4': 'https://www.centralbank.org.bz/home/core-functions/prudential-supervision/credit-unions',
     'BZ CBBEL 5': 'https://www.centralbank.org.bz/home/core-functions/prudential-supervision/payment-service-providers',
}
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

Typology={
       regulatorName + ' 1': 'Licensed Commercial Banks',
       regulatorName + ' 2': 'Licensed International Banks',
       regulatorName + ' 3': 'Other Licensed Financial Institutions',
       regulatorName + ' 4': 'Registered Credit Unions',
       regulatorName + ' 5': 'Approved Money Transfer Service Providers',

}

processdate=now.strftime('%Y-%m-%d')



In [8]:

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



In [ ]:
#------------------------------------------------ Main_Fouction ----------------------------------------

for reg in regdict:
    print('Working with {}'.format(reg))
    #driver.get(regdict[reg])
    response = requests.get(regdict[reg], verify=False, timeout=30)
    data = response.text  
    soup = BeautifulSoup(data, "html.parser")

    if reg == 'BZ CBBEL 1' or reg == 'BZ CBBEL 2' or reg == 'BZ CBBEL 4':
        contents = soup.find('div',class_='accordion sf_cols')
        lis = contents.find('div',class_='item-list horizontal').find_all('div')
        for li in lis:
            name = li.find('h3').text
            address = li.find('address').text
            ps = li.find('p')
            tel_index = ps.text.find('Tel')
            email_index = ps.text.find('Email:')
            web_index = ps.text.find('Web:')
            web_index_end = ps.text.find('.com')
            
            raw_phone  = ps.text[tel_index + 4:web_index].strip()
            phone_only =  re.match(r'^[^A-Za-z]*', raw_phone).group().strip()
            phone_ = phone_only.replace(':', '')
            web_ = ps.text[web_index + 4:web_index_end+4].strip()
            sqldict['Name'].append(name)
            sqldict['Address_1'].append(address)
            sqldict['Phone'].append(phone_)
            sqldict['Website'].append(web_)
            sqldict['ListProcessDate'].append(processdate)		
            sqldict['RegCtry'].append('BZ')		
            sqldict['Cntry'].append('BZ')		
            sqldict['RegCode'].append('CBBEL')		
            sqldict['ListCode'].append(reg.split(' ')[-1])		
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(Typology[reg])
            sqldict = bourange_same_length_array(sqldict)

    elif reg == 'BZ CBBEL 3':
        contents = soup.find('div',class_='accordion sf_cols')
        lis = contents.find('div',class_='item-list horizontal').find_all('div')
        for li in lis:
            #print(li)
            name = li.find('h3').text
            ps = li.find_all('p')
            address = ps[0].text
            address = address.replace(':','')
            sqldict['Name'].append(name)
            sqldict['Address_1'].append(address)    
            sqldict['ListProcessDate'].append(processdate)		
            sqldict['RegCtry'].append('BZ')		
            sqldict['Cntry'].append('BZ')		
            sqldict['RegCode'].append('CBBEL')		
            sqldict['ListCode'].append(reg.split(' ')[-1])		
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(Typology[reg])
            sqldict = bourange_same_length_array(sqldict)
    elif reg == 'BZ CBBEL 5':
        contents = soup.find('div',class_='accordion sf_cols')
        lis = contents.find('div',class_='item-list horizontal').find_all('div')
        for li in lis:
            name = li.find('h3').text
            ps = li.find_all('p')
            total_address = ''
            for p in ps:
                total_address += p.get_text(strip=True)
            total_address = total_address.replace('Agent','')

            sqldict['Name'].append(name)
            sqldict['Address_1'].append(total_address) 
            sqldict['ListProcessDate'].append(processdate)		
            sqldict['RegCtry'].append('BZ')		
            sqldict['Cntry'].append('BZ')		
            sqldict['RegCode'].append('CBBEL')		
            sqldict['ListCode'].append(reg.split(' ')[-1])		
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(Typology[reg])
            sqldict = bourange_same_length_array(sqldict)



Working with BZ CBBEL 1


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.centralbank.org.bz'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Working with BZ CBBEL 2


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.centralbank.org.bz'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Working with BZ CBBEL 3


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.centralbank.org.bz'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Working with BZ CBBEL 4


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.centralbank.org.bz'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Working with BZ CBBEL 5


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.centralbank.org.bz'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [10]:
		
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()
    

C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_23828\4202423384.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [11]:
df.to_excel(filename)